# hy-axis L=4 — S2 and V-score along the pure-h_y line

Stage-1 cold campaign (2026-08-31, branch `feat/hy-axis-L4`): 16 cold points
hx=hz=0, h_y = 0.0…1.5 (Δ=0.1), 500 SR steps, dual-basis complex
`ToricCNN_gridinv` (nh 4→8, inv 8-8, k=3), snapshots every 50.
S2 = central-plaquette Rényi-2 from the **in-job final-state replay**
(`*.finaleval_electric.json`, step 500); V-score from each run's pooled final
eval. Exact anchors: E0(h=0) = −172, S2(h=0) = 3·ln2.
Headline: first-order transition between h_y = 1.2 and 1.3 (S2 plateau
collapse + dE/dh_y slope jump 21→190 + N⟨σy⟩ jump 31→134).

In [ ]:
# §1 CONFIG + loaders
import glob, json, os
import numpy as np
import matplotlib.pyplot as plt

ROOT = os.path.abspath(os.path.join(os.getcwd(), "..", ".."))
DATA = os.path.join(ROOT, "results", "hy_axis_L4", "cold", "L4")
FIGS = os.path.join(os.getcwd(), "..", "figs")
Ls = [4]

plt.rcParams.update({"figure.dpi": 120, "font.size": 11,
                     "axes.spines.top": False, "axes.spines.right": False,
                     "axes.grid": True, "grid.alpha": 0.3})
COL = dict(zip(Ls, plt.cm.plasma(np.linspace(0, 0.85, len(Ls)))))

S2_EXACT_H0 = 3 * np.log(2)
TRANSITION_WIN = (1.2, 1.3)          # S2 collapse + energy-slope jump window

rows = []
for f in sorted(glob.glob(f"{DATA}/*_k3.json")):      # seed-0 campaign points
    m = json.load(open(f))
    c, o = m["config"], m["observables"]
    if c["hy"] < 0:                                    # TR pairs: not this curve
        continue
    fe = f[:-len(".json")] + ".finaleval_electric.json"
    s = json.load(open(fe))["series"][-1]
    assert s["step"] == c["n_iter"], (f, s["step"])
    rows.append(dict(hy=c["hy"], E=o["E0"], V=o["Vscore"],
                     S2=s["S2"], S2e=s["S2_err"], div=m["diverged"]))
rows.sort(key=lambda r: r["hy"])
hy   = np.array([r["hy"] for r in rows])
S2   = np.array([r["S2"] for r in rows])
S2e  = np.array([r["S2e"] for r in rows])
V    = np.array([r["V"] for r in rows])
assert not any(r["div"] for r in rows)
print(f"{len(rows)} points, hy = {hy.min()} .. {hy.max()}")

## §2 S2(h_y) — central-plaquette Rényi-2 at end of training

Topological plateau at 3·ln2 through h_y ≤ 1.1, collapse to ≈0.13–0.18 beyond
h_y = 1.3. The h_y=0.5 point (2.27) pokes above the plateau — flagged as
likely under-converged noise (S2 error bars are optimistic at h_y ≠ 0: no
phase-coherence diagnostic); to be re-checked in the snapshot-series replay.

In [ ]:
fig, ax = plt.subplots(figsize=(6.4, 4.2))
ax.axvspan(*TRANSITION_WIN, color="0.85", alpha=0.5, zorder=0)
ax.axhline(S2_EXACT_H0, color="k", ls=":", lw=1,
           label=r"$3\ln 2$ (exact, $h=0$)")
ax.errorbar(hy, S2, yerr=S2e, fmt="o-", ms=4, lw=1.2, color=COL[4],
            mec="k", mew=0.4, ecolor="0.5", capsize=2, zorder=3, label="L=4")
ax.set(xlabel=r"$h_y$", ylabel=r"$S_2$ (central plaquette)")
ax.legend(fontsize=8, loc="upper left")
fig.tight_layout()
plt.show()
# fig.savefig(f"{FIGS}/hy_axis_L4_S2.png", dpi=300, bbox_inches="tight")

## §3 V-score(h_y)

In the topological phase the V-score tracks the sign-structure cost
≈ 0.5·h_y² (dashed); past the transition it drops an order of magnitude —
the polarized state's simpler sign structure. (h_y=0 sits at ~1e-11: the
dual net is exact there.)

In [ ]:
fig, ax = plt.subplots(figsize=(6.4, 4.2))
ax.axvspan(*TRANSITION_WIN, color="0.85", alpha=0.5, zorder=0)
hh = np.linspace(1e-3, hy.max(), 200)
ax.plot(hh, 0.5 * hh**2, "--", color="0.4", lw=1.4,
        label=r"$0.5\,h_y^2$ (sign-structure cost)")
ax.plot(hy, V, "o-", ms=4, lw=1.2, color=COL[4], mec="k", mew=0.4,
        zorder=3, label="L=4")
ax.set(xlabel=r"$h_y$", ylabel="V-score", yscale="log")
ax.legend(fontsize=8, loc="upper left")
fig.tight_layout()
plt.show()
# fig.savefig(f"{FIGS}/hy_axis_L4_vscore.png", dpi=300, bbox_inches="tight")